In [4]:
##Imports

from playwright.async_api import async_playwright
import pandas as pd
import asyncio

In [14]:
##Scrapes from enforcementtracker

async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto("https://www.enforcementtracker.com")
        await asyncio.sleep(3)

        while True:
            before = await page.locator("table tr").count()
            await page.evaluate("document.querySelector('button.et-btn-primary').dispatchEvent(new MouseEvent('click', {bubbles: true}))")
            await asyncio.sleep(2)
            after = await page.locator("table tr").count()
            if after == before:
                break

        rows = await page.locator("table tr").all()
        data = []
        for row in rows:
            cells = await row.locator("td").all_text_contents()
            if cells:
                data.append(cells)

        headers = await page.locator("table tr th").all_text_contents()
        df = pd.DataFrame(data, columns=headers)
        print(df.head())
        print(df.shape)
        await browser.close()
        return df

df = await scrape()


                                                ETid  \
0  \n                                ETid-3195\n ...   
1  \n                                ETid-3194\n ...   
2  \n                                ETid-3193\n ...   
3  \n                                ETid-3192\n ...   
4  \n                                ETid-3191\n ...   

                                             Country  \
0  \n                                \n          ...   
1  \n                                \n          ...   
2  \n                                \n          ...   
3  \n                                \n          ...   
4  \n                                \n          ...   

                                           Authority        Date    Fine (€)  \
0  Romanian National Supervisory Authority for Pe...  2026-06-12       5,000   
1  Hungarian National Authority for Data Protecti...  2026-05-29      70,300   
2     Norwegian Supervisory Authority (Datatilsynet)  2026-06-01   1,820,000   
3     

In [23]:
##Cleans

df = df.apply(lambda x: x.str.replace(r'\s+', ' ', regex=True).str.strip() if x.dtype == "object" else x)
df = df.iloc[:, :8]
df['ETid'] = df['ETid'].str.extract(r'(ETid-\d+)')
df['Country'] = df['Country'].str.replace(r'\s+', ' ', regex=True).str.strip().str.split(' ').str[-1]
print(df.head())

        ETid  Country                                          Authority  \
0  ETid-3195  Romania  Romanian National Supervisory Authority for Pe...   
1  ETid-3194  Hungary  Hungarian National Authority for Data Protecti...   
2  ETid-3193   Norway     Norwegian Supervisory Authority (Datatilsynet)   
3  ETid-3192    Spain           Spanish Data Protection Authority (aepd)   
4  ETid-3191    Spain           Spanish Data Protection Authority (aepd)   

         Date    Fine (€)  Controller / Processor  \
0  2026-06-12       5,000  Națională Poșta Română   
1  2026-05-29      70,300              Blikk Kft.   
2  2026-06-01   1,820,000               Elkjøp AS   
3  2025-04-07  14,400,000  AMADEUS IT GROUP, S.A.   
4  2026-04-22     300,000       KONECTA BTO, S.L.   

                             Sector  \
0         Transportation and Energy   
1  Media, Telecoms and Broadcasting   
2             Industry and Commerce   
3         Transportation and Energy   
4             Industry and Co

In [24]:
##Saves to .csv, to be tweaked in EDA

df.to_csv("enforcement_tracker_clean.csv", index=False)
print(df.shape)
print(df.dtypes)

(3195, 8)
ETid                         str
Country                   object
Authority                    str
Date                         str
Fine (€)                     str
Controller / Processor       str
Sector                       str
Article(s)                   str
dtype: object
